# TalentDesk, Module 2 Section 1 Lab (Exercise): Tool Descriptions, Scoping, and tool_choice

A hands-on exercise on **tool selection**, built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`). It combines the two M2S1 skills: writing **tool descriptions** so the model
routes cleanly, measured before and after (Lab 1), and controlling **which tools an agent has**
(scoping) plus **how hard it must use them** with `tool_choice`, including forcing a tool for
guaranteed structured output (Lab 2). You fill in four short `TODO` blocks; everything else is
provided. An offline keyword-overlap router lets you watch routing improve without a key, and a full
solution is at the end.

## The real-world scenario

TalentDesk's recruiter agent now has several read tools about a candidate: their pipeline stage,
their interview schedule, their application details, and their offer eligibility. The model does not
read your code to choose one; it reads the **descriptions**. If all four say "get candidate
information," the model has no honest way to pick, so it routes at random and sometimes runs the
wrong lookup. The fix is not more code; it is sharper descriptions with clear boundaries.

Two more levers matter once the toolbox grows. **Scoping** decides which tools a given agent even
has: a scheduling coordinator that *cannot* check offer eligibility is a risk you simply removed.
And `tool_choice` decides how hard the model must use a tool, which is how you guarantee a
machine-readable result for a step that must not fail.

The question this lab answers: **how much does description wording change which tool the model picks,
how do you give each agent only the tools its role needs, and how do you force a tool when a step
must produce structure?**

## Objectives

- See that the **description** is the primary signal the model uses to select a tool, and rewrite
  overlapping descriptions to include **boundaries** (slice, input, example queries, when NOT to
  use).
- **Measure** selection with an overlap score and an accuracy test, before and after refinement.
- Assign **full vs scoped** tool sets per role so out-of-role actions become impossible.
- Use `tool_choice` (**auto**, **any**, **forced**) and force a tool to **guarantee structured
  output**.

## The outcome you should reach

By the end you will have:

- an overlap score that is high for vague tools and low for refined ones;
- refined descriptions that route a set of ambiguous queries correctly where the vague ones do not;
- scoped tool sets where a scheduling agent structurally cannot reach the offer tool;
- and a forced output tool that returns a guaranteed structured payload every time.

Target time: **20 to 30 minutes.** Four small `TODO` blocks. The overlap score, the scoped sets, and
the output-tool schema are testable offline; a mock router shows the routing gain without a key, and
the live selection test needs a real key.

## How to run

Run top to bottom. The overlap score, scoped sets, and output schema are pure Python and run
anywhere; an offline **keyword-overlap router** shows routing improve from vague to refined. To run
the real selection test, paste a real key into **Setup 2/3** and re-run from the top. This lab uses
the base Messages API so you can read exactly which tool the model chose.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic SDK**, because
tool selection and `tool_choice` are properties of one Messages API call: you pass `tools` and read
back which one the model picked.

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports, the model, and the `RUN_LIVE` switch so live calls fire only with a
real key.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import re                                       # tokenise descriptions for the overlap score
import json                                     # print structured tool inputs
import itertools                                # pair tools up for the overlap score
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (offline router shows the routing gain)")

**This cell:** the shared input schema and a small `words()` helper (provided). All tools take
one `candidate_id`, so the *only* thing that distinguishes them is the description. `words()` reduces
a description to its content words, which both the overlap score and the offline router use.

In [ ]:
# ===== SETUP 3/3 - the shared schema and a word helper (provided) =====
ARG = {"type": "object",                          # every tool takes one string candidate_id
       "properties": {"candidate_id": {"type": "string"}},
       "required": ["candidate_id"]}

STOP = {"the", "a", "an", "of", "for", "to", "or", "and", "use", "do", "not", "by",
        "on", "in", "with", "is", "any", "one", "return", "get", "this", "that"}   # true filler only

def words(text):                                  # text -> its set of content words
    return {w for w in re.findall(r"[a-z']+", text.lower()) if w not in STOP and len(w) > 2}

print("shared schema and words() ready")

### The description is the decision signal

The model chooses a tool by matching the request against each tool's **description**, not its name or
code. A good description draws a clear boundary: what the tool is for, the input it expects, example
queries that should trigger it, and, just as important, when NOT to use it. Overlapping descriptions
blur those boundaries and cause mis-routing.

---

### 🎯 Part A - sharpen the descriptions, measure the gain

**This cell:** the **vague** tool set (provided). Every description is a generic "get candidate
information," so they overlap heavily and the model cannot tell them apart.

In [ ]:
# ===== the four vague, overlapping tools (provided) =====
NAMES = ["get_candidate_stage", "get_interview_schedule", "get_application_details", "check_offer_eligibility"]

VAGUE = [{"name": n, "description": "Get candidate information.", "input_schema": ARG} for n in NAMES]
for t in VAGUE:
    print(f"  {t['name']:24} {t['description']}")

**TODO 1 (about 4 minutes).** Complete `overlap_score()`, the pure-Python ambiguity proxy. For
each pair of tools, compute the Jaccard overlap of their description word sets (shared words divided
by total words), then return the average, rounded to 3 places.

In [ ]:
# ===== TODO 1 - measure description overlap (offline proxy for ambiguity) =====
def overlap_score(tools):                         # average pairwise word overlap (Jaccard)
    sets = [words(t["description"]) for t in tools]
    pairs = list(itertools.combinations(sets, 2)) #   every pair of tools
    # 👉 TODO 1a: jac = [len(a & b) / len(a | b) for a, b in pairs if (a | b)]
    # 👉 TODO 1b: return round(sum(jac) / len(jac), 3) if jac else 0.0
    return 0.0                                     #   replace this line

print("vague overlap:", overlap_score(VAGUE), "  (higher = more ambiguous)")

**Self-check (offline).** The vague set is identical text, so its overlap should be very
high.

In [ ]:
# ===== self-check for TODO 1 =====
assert overlap_score(VAGUE) > 0.8, "identical vague descriptions should score near 1.0"
print("TODO 1 check passed:", overlap_score(VAGUE))

**TODO 2 (about 6 minutes).** Write a **refined** description for each tool in
`REFINED_DESCRIPTIONS`. Each should state its slice, an example query or two, and when NOT to use it.
Make sure the example queries use the same words a real request would (that is what routes cleanly).
The four tools own: pipeline **stage**, **interview** schedule, application **resume** details, and
**offer** eligibility.

In [ ]:
# ===== TODO 2 - write refined descriptions with clear boundaries =====
REFINED_DESCRIPTIONS = {
    # 👉 fill in each description. A worked pattern for the first one:
    "get_candidate_stage": "",       # e.g. pipeline stage: 'what stage is C1 at', 'where is C1 in the process'. Not for interviews/resume/offers.
    "get_interview_schedule": "",    # interview time/logistics: 'when is C1's interview scheduled'. Not for stage/resume/offers.
    "get_application_details": "",   # resume contents, skills, experience: 'what skills and experience are on C1's resume'. Not for stage/interview/offers.
    "check_offer_eligibility": "",   # offer eligibility and band: 'is C1 eligible for an offer within band'. Not for stage/interview/resume.
}
REFINED = [{"name": n, "description": REFINED_DESCRIPTIONS[n], "input_schema": ARG} for n in NAMES]
print("refined overlap:", overlap_score(REFINED), "  (lower = clearer boundaries)")

**Self-check (offline).** The refined set should overlap far less than the vague one.

In [ ]:
# ===== self-check for TODO 2 =====
assert all(REFINED_DESCRIPTIONS[n].strip() for n in NAMES), "write a description for every tool"
assert overlap_score(REFINED) < overlap_score(VAGUE), "refined descriptions should overlap less"
print("TODO 2 check passed: overlap dropped from", overlap_score(VAGUE), "to", overlap_score(REFINED))

**This cell:** an offline **keyword-overlap router** and the labelled query set (provided). The
router picks the tool whose description shares the most words with the query, exactly the kind of
match the model makes. With vague descriptions the queries tie and mis-route; with your refined
descriptions they should route correctly.

In [ ]:
# ===== offline routing accuracy: vague vs refined (provided) =====
EVAL = [                                           # (query, the tool that SHOULD be picked)
    ("What stage is candidate C1 at in the pipeline?",        "get_candidate_stage"),
    ("When is C1's interview scheduled?",                     "get_interview_schedule"),
    ("What skills and experience are on C1's resume?",        "get_application_details"),
    ("Is C1 eligible for an offer within band?",              "check_offer_eligibility"),
    ("Where is candidate C2 in the process right now?",       "get_candidate_stage"),
]

def mock_pick_tool(query, tools):                  # pick the description with the most shared words
    q = words(query)
    best, best_score = tools[0]["name"], -1
    for t in tools:
        s = len(q & words(t["description"]))       #   shared content words with this description
        if s > best_score:
            best_score, best = s, t["name"]        #   ties keep the first (a vague mis-route)
    return best

def mock_accuracy(tools, label):
    correct = 0
    for query, expected in EVAL:
        got = mock_pick_tool(query, tools)
        correct += (got == expected)
    print(f"  {label:8} routing accuracy: {correct}/{len(EVAL)}")
    return correct

av = mock_accuracy(VAGUE, "vague")
ar = mock_accuracy(REFINED, "refined")
assert ar > av, "refined descriptions should route more queries correctly than vague ones"
print("refined beats vague offline:", ar, ">", av)

**This cell:** the **live** selection test (provided). `pick_tool` uses
`tool_choice={"type":"any"}` so the model must pick one tool, letting us read the choice. Offline it
prints the expected outcome the mock router just demonstrated.

In [ ]:
# ===== live selection accuracy: vague vs refined =====
def pick_tool(query, tools):                       # query -> the chosen tool name (live)
    client = anthropic.Anthropic()
    r = client.messages.create(model=MODEL, max_tokens=200, tools=tools,
                               tool_choice={"type": "any"},
                               messages=[{"role": "user", "content": query}])
    for b in r.content:
        if b.type == "tool_use":
            return b.name
    return "(none)"

def live_accuracy(tools, label):
    correct = 0
    for query, expected in EVAL:
        got = pick_tool(query, tools)
        ok = got == expected
        correct += ok
        print(f"  {label:8} {query[:34]!r:36} picked={got:24} {'ok' if ok else 'X'}")
    print(f"  {label} accuracy: {correct}/{len(EVAL)}\n")
    return correct

if RUN_LIVE:
    print("vague:");   live_accuracy(VAGUE, "vague")
    print("refined:"); live_accuracy(REFINED, "refined")
else:
    print("[offline] expected live: refined routes more (often all) queries correctly than vague,")
    print("          matching the mock router above. Only the description text changed.")

---

### 🎯 Part B - scope the tools, then control the choice

Two separate controls: **scoping** decides *availability* (a tool an agent does not have cannot be
chosen), and **`tool_choice`** decides *obligation* (`auto` may answer with text, `any` forces some
tool, a forced tool forces that exact one).

**TODO 3 (about 4 minutes).** Build the scoped tool sets from the refined tools. A **scheduling
coordinator** should see only stage and interview tools; a **screening reviewer** should see only
application-details and offer-eligibility tools. Scoping out `check_offer_eligibility` is what makes
it impossible for the scheduling agent to reach it.

In [ ]:
# ===== TODO 3 - scoped tool sets, one per role =====
by_name = {t["name"]: t for t in REFINED}          # look tools up by name

FULL = list(REFINED)                               # every tool (the risky default)

# 👉 TODO 3a: SCHEDULING_SCOPED = the stage and interview tools only
SCHEDULING_SCOPED = []                             # replace: [by_name["get_candidate_stage"], by_name["get_interview_schedule"]]

# 👉 TODO 3b: SCREENING_SCOPED = the application-details and offer-eligibility tools only
SCREENING_SCOPED = []                              # replace: [by_name["get_application_details"], by_name["check_offer_eligibility"]]

print("scheduling agent can use:", [t["name"] for t in SCHEDULING_SCOPED])
print("screening agent can use: ", [t["name"] for t in SCREENING_SCOPED])

**Self-check (offline).** Confirms the scoping guarantee: the scheduling agent structurally
cannot reach the offer tool.

In [ ]:
# ===== self-check for TODO 3 =====
sched = {t["name"] for t in SCHEDULING_SCOPED}
screen = {t["name"] for t in SCREENING_SCOPED}
assert sched == {"get_candidate_stage", "get_interview_schedule"}
assert "check_offer_eligibility" not in sched, "the scheduling agent must not be able to check offers"
assert screen == {"get_application_details", "check_offer_eligibility"}
print("TODO 3 checks passed: scheduling agent cannot reach the offer tool")

**This cell:** the `run()` helper and the two live `tool_choice` demos (provided). `auto` lets a
greeting get a text reply; `any` forces a tool even on that greeting. Offline these print the expected
behaviour.

In [ ]:
# ===== the one-call helper and auto vs any (provided) =====
def run(query, tools, choice):                     # (query, tools, tool_choice) -> what happened
    client = anthropic.Anthropic()
    r = client.messages.create(model=MODEL, max_tokens=300, tools=tools,
                               tool_choice=choice,
                               messages=[{"role": "user", "content": query}])
    for b in r.content:
        if b.type == "tool_use":
            return ("tool", b.name, b.input)
    return ("text", "".join(b.text for b in r.content if b.type == "text"), None)

if RUN_LIVE:
    print("auto on a greeting ->", run("Hi, thanks for the help earlier!", FULL, {"type": "auto"})[:2])
    print("any  on a greeting ->", run("Hi, thanks for the help earlier!", FULL, {"type": "any"})[:2])
else:
    print("[offline] expected: auto -> a text reply and NO tool; any -> forced to pick some tool.")

**TODO 4 (about 5 minutes).** Build the **output tool** `record_screening`, whose `input_schema`
is the structured result you want back, then set `FORCE` to force exactly that tool. Forcing an output
tool is how you guarantee structured output: the model must call it, so its `input` comes back as a
JSON payload matching your schema.

The payload needs: `candidate_id` (string), `recommendation` (enum: advance, hold, reject), and
`priority` (enum: low, medium, high), all required.

In [ ]:
# ===== TODO 4 - a forced output tool for guaranteed structured output =====
record_screening = {
    "name": "record_screening",
    "description": "Record the screening result for a candidate.",
    "input_schema": {
        "type": "object",
        # 👉 TODO 4a: add "properties" with candidate_id (string),
        #    recommendation (enum advance/hold/reject), priority (enum low/medium/high)
        # 👉 TODO 4b: add "required": ["candidate_id", "recommendation", "priority"]
    },
}

# 👉 TODO 4c: FORCE the model to call exactly this tool
FORCE = {"type": "auto"}   # replace with {"type": "tool", "name": "record_screening"}

print("output tool ready:", record_screening["name"])

**Self-check (offline).** Confirms the schema shape and the forced choice.

In [ ]:
# ===== self-check for TODO 4 =====
props = record_screening["input_schema"].get("properties", {})
assert set(props) == {"candidate_id", "recommendation", "priority"}, "define all three fields"
assert props["recommendation"]["enum"] == ["advance", "hold", "reject"]
assert props["priority"]["enum"] == ["low", "medium", "high"]
assert record_screening["input_schema"].get("required") == ["candidate_id", "recommendation", "priority"]
assert FORCE == {"type": "tool", "name": "record_screening"}, "force this exact tool"
print("TODO 4 checks passed")

**This cell:** the **forced structured output** run (provided). Forcing `record_screening` means
the model must call it, so the payload is a guaranteed dict, no prose to parse. Offline it prints an
example of the shape.

In [ ]:
# ===== forced tool: guaranteed structured output =====
ticket = "C1 has 5 years and clears the bar; move them forward, it is urgent."
if RUN_LIVE:
    kind, name, data = run(ticket, [record_screening], FORCE)
    print("forced ->", kind, "| tool:", name)
    print("structured payload:", json.dumps(data))
else:
    print("[offline] expected: a call to record_screening with, e.g.,")
    print('          {"candidate_id":"C1","recommendation":"advance","priority":"high"}')

**This cell:** how these map to the **Agent SDK** (provided). Scoping is productised as
`AgentDefinition(tools=[...])` per subagent; `tool_choice` is not exposed by the Agent SDK, so when
you need forced or structured output you reach for the base Messages API, exactly as above.

In [ ]:
# ===== how these map to the Agent SDK (provided) =====
print("descriptions -> same text in @tool(...) or AgentDefinition(description=...)")
print("scoping      -> AgentDefinition(tools=[...])   # each subagent sees only its role's tools")
print("forcing      -> base Messages API tool_choice  # Agent SDK does not expose tool_choice")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| "get candidate info" on four tools | give each a distinct slice and say what it is for |
| rely on the tool name to disambiguate | put the routing signal in the description, not the name |
| omit when-NOT-to-use guidance | name the neighbours and say which tool to use instead |
| give every agent every tool | scope each agent to its role's tools |
| rely on a prompt to keep an agent in its lane | remove the out-of-role tool from its set entirely |
| parse JSON out of a free-text reply | force an output tool so the payload is structured |
| use `any` everywhere | use `auto` when text is fine; force only when a step must produce structure |

**Lesson:** the model routes on **descriptions**, so overlapping ones mis-route no matter how
good the model is; give each tool a clear boundary and measure with an overlap score and an accuracy
test. Then control tools with two separate levers: **scoping** decides what an agent can do at all,
and **`tool_choice`** decides how hard it must use its tools, with a forced output tool guaranteeing a
structured result for a deterministic step.

---

## Recap - descriptions, scope, and choice

| Lever | In this lab | Course topic |
|---|---|---|
| Description as signal | vague vs refined, overlap score | descriptions guide selection (Lab 1) |
| Boundaries | slice, input, examples, when-not | clear tool boundaries (Lab 1) |
| Measurement | overlap score + accuracy over ambiguous queries | test selection before vs after (Lab 1) |
| Scoping | full vs role-scoped `tools` | out-of-role actions become impossible (Lab 2) |
| tool_choice | auto / any / forced | flexibility vs required vs guaranteed (Lab 2) |
| Forced output tool | `record_screening` schema, forced | deterministic structured output (Lab 2) |

**Try it next:** add a fifth tool that overlaps `get_candidate_stage` and watch the overlap score rise
and the router mis-route until you give it a clear boundary. Then force `record_screening` on a plain
greeting and confirm it still returns a valid structured payload.